In [1]:
from train import train, TrainingConfig
import os
from helpers import load_bpe_tokenization, load_encoding, save_model
import torch
from datasets import StrideDataset
from simpleGPT import SimpleGPTConfig, SimpleGPT
from block import BlockConfig

In [2]:
models_dir_name = models_dir_path = "models"
model_name = "mickiewicz_gpt_020"
tokenizers_dir_name = tokenizers_dir_path = "tokenizers"
tokenizer_name = "tokenizer_mickiewicz_010"
encodings_dir_name = encodings_dir_path = "encodings"
tr_encoding_name = "tr_encoding_mickiewicz_010"
val_encoding_name = "val_encoding_mickiewicz_010"

In [3]:
# Tokenizer
encode, decode, vocab, merges = load_bpe_tokenization(os.path.join(tokenizers_dir_path, tokenizer_name + ".pt"))

In [4]:
tr_encoding = load_encoding(os.path.join(encodings_dir_path, tr_encoding_name + ".pt"))
val_encoding = load_encoding(os.path.join(encodings_dir_path, val_encoding_name + ".pt"))

In [5]:
BLOCK_SIZE = 64
tr_dataset = StrideDataset(tr_encoding, BLOCK_SIZE, stride=BLOCK_SIZE//8)
val_dataset = StrideDataset(val_encoding, BLOCK_SIZE, stride=BLOCK_SIZE//8)

In [6]:
N_BLOCKS = 6
N_EMBD = 256
N_HEADS = 6
ATTENTION_INNER_DIM = 64
FF_EMBD_TO_DIM_RATIO = 4.0
DROPOUT = 0.2
model = SimpleGPT(SimpleGPTConfig(
    vocab_size=len(vocab),
    n_blocks=N_BLOCKS,
    block_config=BlockConfig(
        n_heads=N_HEADS,
        n_embd=N_EMBD,
        attention_inner_dim=ATTENTION_INNER_DIM,
        ff_embedding_to_dim_ratio=FF_EMBD_TO_DIM_RATIO,
        dropout=DROPOUT
    )
))
print(f"Model has: {sum([p.numel() for p in model.parameters()])} learnable parameters")

Model has: 7614972 learnable parameters


In [7]:
BATCH_SIZE = 512
NUM_EPOCHS = 15
LR = 1e-4
INFO_INTERVAL = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
training_config = TrainingConfig(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    info_interval=INFO_INTERVAL,
    device=DEVICE
)
train(model, tr_dataset, val_dataset, training_config)

Using device: cuda
Train dataset length: 40316 | Val dataset length: 4472
Epoch 0 | Train loss: 8.5559 | Val loss: 8.5645
Epoch 1 | Train loss: 7.8091 | Val loss: 7.5492
Epoch 2 | Train loss: 7.5687 | Val loss: 7.5438
Epoch 3 | Train loss: 7.5579 | Val loss: 7.5378
Epoch 4 | Train loss: 7.5481 | Val loss: 7.5330
Epoch 5 | Train loss: 7.5254 | Val loss: 7.4368
Epoch 6 | Train loss: 7.1644 | Val loss: 6.8902
Epoch 7 | Train loss: 6.7089 | Val loss: 6.6522
Epoch 8 | Train loss: 6.5026 | Val loss: 6.5197
Epoch 9 | Train loss: 6.3423 | Val loss: 6.4100
Epoch 10 | Train loss: 6.1942 | Val loss: 6.3114
Epoch 11 | Train loss: 6.0595 | Val loss: 6.2161
Epoch 12 | Train loss: 5.9442 | Val loss: 6.1604
Epoch 13 | Train loss: 5.8450 | Val loss: 6.0990
Epoch 14 | Train loss: 5.7616 | Val loss: 6.0532
Epoch 15 | Train loss: 5.6835 | Val loss: 6.0231


In [8]:
model.to('cpu')
save_model(model, model.config, os.path.join(models_dir_path, model_name + ".pt"))

[save_model] Model saved to models/mickiewicz_gpt_020.pt


In [ ]:
INFO_INTERVAL = 5
NUM_EPOCHS = 50
training_config.info_interval = INFO_INTERVAL
training_config.num_epochs = NUM_EPOCHS
train(model, tr_dataset, val_dataset, training_config)

Using device: cuda
Train dataset length: 40316 | Val dataset length: 4472
Epoch 0 | Train loss: 5.5034 | Val loss: 6.0231
